# Preprocessing: Annotations → Clips

This notebook handles two preprocessing steps for Phase 2A (no_hardware):

1. **Annotation conversion** — reads Label Studio JSON exports and produces a unified `annotations.csv`
2. **Clip extraction** — uses the CSV to slice individual punch clips from the full-video pose files

**Outputs:**
- `data/metadata/no_hardware/annotations.csv` — master annotation table
- `data/clips/no_hardware/{class}/{clip_id}.npy` — individual clip files for training

In [13]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("../../")  # adjust if notebook lives elsewhere

ANNOTATIONS_ROOT = PROJECT_ROOT / "data" / "annotations" / "no_hardware"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "no_hardware"
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "no_hardware"
CLIPS_DIR = PROJECT_ROOT / "data" / "clips" / "no_hardware"

ANNOTATIONS_CSV = METADATA_DIR / "annotations.csv"

METADATA_DIR.mkdir(parents=True, exist_ok=True)
CLIPS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Annotations root: {ANNOTATIONS_ROOT}")
print(f"Processed root:   {PROCESSED_ROOT}")
print(f"Output CSV:       {ANNOTATIONS_CSV}")
print(f"Output clips:     {CLIPS_DIR}")

Annotations root: ..\..\data\annotations\no_hardware
Processed root:   ..\..\data\processed\no_hardware
Output CSV:       ..\..\data\metadata\no_hardware\annotations.csv
Output clips:     ..\..\data\clips\no_hardware


## Section A — Annotation Conversion

Reads Label Studio JSON exports from `data/annotations/no_hardware/{subject}/labelstudio/*.json` and builds a unified DataFrame of all clips with metadata.

In [14]:
def parse_video_filename(filename: str) -> dict:
    """Parse 'cross_1m_left.mp4' (with optional Label Studio hash prefix) into {class, distance_m, hand}."""
    stem = Path(filename).stem
    
    # Strip Label Studio hash prefix if present (e.g. "11506747-cross_1m_left" → "cross_1m_left")
    if "-" in stem:
        prefix, _, rest = stem.partition("-")
        # Hash prefix is hex characters only
        if all(c in "0123456789abcdef" for c in prefix.lower()):
            stem = rest
    
    match = re.match(r"(jab|cross|hook|uppercut)_(\d+)m_(left|right)", stem)
    if not match:
        raise ValueError(f"Could not parse filename: {filename}")
    return {
        "class": match.group(1),
        "distance_m": int(match.group(2)),
        "hand": match.group(3),
    }

# Test with both formats
print(parse_video_filename("cross_1m_left.mp4"))
print(parse_video_filename("11506747-cross_1m_left.mp4"))

{'class': 'cross', 'distance_m': 1, 'hand': 'left'}
{'class': 'cross', 'distance_m': 1, 'hand': 'left'}


In [15]:
EXCLUDED_CLASSES = {"clap"}

def load_labelstudio_annotations(annotations_root: Path) -> list[dict]:
    """Walk through subject folders and load all Label Studio JSON exports."""
    records = []
    subject_dirs = sorted(annotations_root.glob("subject*"))
    print(f"Found {len(subject_dirs)} subject folders")
    
    for subject_dir in subject_dirs:
        subject_id = subject_dir.name
        labelstudio_dir = subject_dir / "labelstudio"
        
        if not labelstudio_dir.exists():
            print(f"  Skipping {subject_id} — no labelstudio folder")
            continue
        
        json_files = sorted(labelstudio_dir.glob("*.json"))
        print(f"  {subject_id}: {len(json_files)} JSON files")
        
        for json_path in json_files:
            with open(json_path) as f:
                data = json.load(f)
            
            tasks = data if isinstance(data, list) else [data]
            
            for task in tasks:
                video_filename = Path(task.get("data", {}).get("video", "")).name
                
                try:
                    file_info = parse_video_filename(video_filename)
                except ValueError:
                    print(f"    Skipping unparseable filename: {video_filename}")
                    continue
                
                for annotation in task.get("annotations", []):
                    for result in annotation.get("result", []):
                        if result.get("type") != "timelinelabels":
                            continue
                        
                        value = result.get("value", {})
                        labels = value.get("timelinelabels", [])
                        ranges = value.get("ranges", [])
                        
                        if not labels or not ranges:
                            continue
                        
                        label = labels[0]
                        if label in EXCLUDED_CLASSES:
                            continue
                        
                        for r in ranges:
                            records.append({
                                "subject_id": subject_id,
                                "video_filename": video_filename,
                                "class": label,
                                "hand": file_info["hand"],
                                "distance_m": file_info["distance_m"],
                                "start_frame": int(r["start"]),
                                "end_frame": int(r["end"]),
                            })
    return records

records = load_labelstudio_annotations(ANNOTATIONS_ROOT)
print(f"\nTotal annotations loaded: {len(records)}")

Found 7 subject folders
  subject01: 1 JSON files
  subject02: 1 JSON files
  subject03: 1 JSON files
  subject04: 1 JSON files
  subject05: 1 JSON files
  subject06: 1 JSON files
  subject07: 1 JSON files

Total annotations loaded: 1548


In [16]:
def build_annotations_df(records: list[dict]) -> pd.DataFrame:
    """Build the master DataFrame with unique clip_ids."""
    df = pd.DataFrame(records)
    df = df.sort_values(["subject_id", "video_filename", "start_frame"]).reset_index(drop=True)
    
    def make_clip_ids(group):
        subj_num = group["subject_id"].iloc[0].replace("subject", "")
        return [
            f"s{subj_num}_{row['class']}_{row['distance_m']}m_{row['hand']}_c{i+1:03d}"
            for i, (_, row) in enumerate(group.iterrows())
        ]
    
    df["clip_id"] = (
        df.groupby(["subject_id", "video_filename"], group_keys=False)
          .apply(lambda g: pd.Series(make_clip_ids(g), index=g.index))
    )
    
    df["frame_count"] = df["end_frame"] - df["start_frame"] + 1
    
    df = df[[
        "clip_id", "subject_id", "video_filename", "class", "hand",
        "distance_m", "start_frame", "end_frame", "frame_count"
    ]]
    return df

df = build_annotations_df(records)
df.head()

,clip_id,subject_id,video_filename,class,hand,distance_m,start_frame,end_frame,frame_count
0,s01_cross_1m_left_c001,subject01,11506747-cross_1m_left.mp4,cross,left,1,102,123,22
1,s01_cross_1m_left_c002,subject01,11506747-cross_1m_left.mp4,cross,left,1,143,163,21
2,s01_cross_1m_left_c003,subject01,11506747-cross_1m_left.mp4,cross,left,1,186,206,21
3,s01_cross_1m_left_c004,subject01,11506747-cross_1m_left.mp4,cross,left,1,231,250,20
4,s01_cross_1m_left_c005,subject01,11506747-cross_1m_left.mp4,cross,left,1,269,288,20


In [17]:
df.to_csv(ANNOTATIONS_CSV, index=False)
print(f"Saved {len(df)} annotations to {ANNOTATIONS_CSV}\n")

print("=== Class distribution ===")
print(df["class"].value_counts())

print("\n=== Subject distribution ===")
print(df["subject_id"].value_counts())

print("\n=== Distance distribution ===")
print(df["distance_m"].value_counts())

print("\n=== Hand distribution ===")
print(df["hand"].value_counts())

print("\n=== Frame count statistics ===")
print(df["frame_count"].describe())

Saved 1548 annotations to ..\..\data\metadata\no_hardware\annotations.csv

=== Class distribution ===
class
hook        392
uppercut    388
cross       384
jab         384
Name: count, dtype: int64

=== Subject distribution ===
subject_id
subject04    255
subject03    247
subject01    245
subject07    241
subject02    240
subject05    160
subject06    160
Name: count, dtype: int64

=== Distance distribution ===
distance_m
1    776
2    772
Name: count, dtype: int64

=== Hand distribution ===
hand
left     775
right    773
Name: count, dtype: int64

=== Frame count statistics ===
count    1548.000000
mean       25.737080
std         5.038827
min        17.000000
25%        21.000000
50%        27.000000
75%        30.000000
max        39.000000
Name: frame_count, dtype: float64


## Section A.5 — Generate no_punch Annotations

Auto-generate no_punch class annotations from the gaps between punch annotations.

For each video, we find unlabelled frame ranges and sample a few clips of consistent length (~25 frames) from those gaps. Total no_punch clips are subsampled to maintain class balance with the punch classes.

In [18]:
import random

NO_PUNCH_CLIP_LENGTH = 25
GAP_BUFFER = 5  # skip this many frames at the start and end of each gap
TARGET_NO_PUNCH_COUNT = 250
NO_PUNCH_SEED = 42

def strip_labelstudio_prefix(filename: str) -> str:
    """Remove Label Studio hash prefix from filename if present."""
    stem = Path(filename).stem
    suffix = Path(filename).suffix
    
    if "-" in stem:
        prefix, _, rest = stem.partition("-")
        if all(c in "0123456789abcdef" for c in prefix.lower()):
            stem = rest
    
    return stem + suffix

def generate_no_punch_annotations(df_punches: pd.DataFrame, processed_root: Path) -> pd.DataFrame:
    """Generate no_punch clips from gaps between punch annotations."""
    rng = random.Random(NO_PUNCH_SEED)
    candidates = []
    
    # Group by video to compute gaps within each
    for (subject_id, video_filename), group in df_punches.groupby(["subject_id", "video_filename"]):
        # Get total video length from the pose file
        clean_filename = strip_labelstudio_prefix(video_filename)
        video_stem = Path(clean_filename).stem
        pose_path = processed_root / subject_id / f"{video_stem}_pose_norm.npy"
        
        if not pose_path.exists():
            print(f"  Skipping {video_filename}: pose file not found")
            continue
        
        total_frames = np.load(pose_path).shape[0]
        
        # Sort punches by start_frame and build sorted intervals
        sorted_punches = group.sort_values("start_frame")
        intervals = [(row["start_frame"], row["end_frame"]) 
                     for _, row in sorted_punches.iterrows()]
        
        # Identify all gaps: head, between punches, tail
        gaps = []
        # Head gap
        if intervals[0][0] > 0:
            gaps.append((0, intervals[0][0] - 1))
        # Inter-punch gaps
        for i in range(len(intervals) - 1):
            gap_start = intervals[i][1] + 1
            gap_end = intervals[i + 1][0] - 1
            if gap_end > gap_start:
                gaps.append((gap_start, gap_end))
        # Tail gap
        if intervals[-1][1] < total_frames - 1:
            gaps.append((intervals[-1][1] + 1, total_frames - 1))
        
        # Sample clips from each gap
        hand = group["hand"].iloc[0]
        distance_m = group["distance_m"].iloc[0]
        
        for gap_start, gap_end in gaps:
            # Apply buffer
            effective_start = gap_start + GAP_BUFFER
            effective_end = gap_end - GAP_BUFFER
            effective_length = effective_end - effective_start + 1
            
            if effective_length < NO_PUNCH_CLIP_LENGTH:
                continue
            
            # How many clips fit?
            max_clips = effective_length // NO_PUNCH_CLIP_LENGTH
            n_clips = min(max_clips, 2)  # up to 2 clips per gap
            
            for _ in range(n_clips):
                max_start = effective_end - NO_PUNCH_CLIP_LENGTH + 1
                clip_start = rng.randint(effective_start, max_start)
                clip_end = clip_start + NO_PUNCH_CLIP_LENGTH - 1
                
                candidates.append({
                    "subject_id": subject_id,
                    "video_filename": video_filename,
                    "class": "no_punch",
                    "hand": hand,
                    "distance_m": distance_m,
                    "start_frame": clip_start,
                    "end_frame": clip_end,
                })
    
    print(f"Generated {len(candidates)} candidate no_punch clips")
    
    # Subsample to target count
    if len(candidates) > TARGET_NO_PUNCH_COUNT:
        rng.shuffle(candidates)
        candidates = candidates[:TARGET_NO_PUNCH_COUNT]
        print(f"Subsampled to {len(candidates)} clips (seed={NO_PUNCH_SEED})")
    
    # Build DataFrame with clip_ids
    df_no_punch = pd.DataFrame(candidates)
    df_no_punch = df_no_punch.sort_values(
        ["subject_id", "video_filename", "start_frame"]
    ).reset_index(drop=True)
    
    # Generate clip_ids
    def make_clip_ids(group):
        """Generate globally unique clip_ids per subject."""
        return [
            f"s{group['subject_id'].iloc[0].replace('subject', '')}_no_punch_{row['distance_m']}m_{row['hand']}_c{i+1:03d}"
            for i, (_, row) in enumerate(group.iterrows())
        ]

    df_no_punch["clip_id"] = (
        df_no_punch.groupby(["subject_id"], group_keys=False)
        .apply(lambda g: pd.Series(make_clip_ids(g), index=g.index))
    )
    
    df_no_punch["frame_count"] = df_no_punch["end_frame"] - df_no_punch["start_frame"] + 1
    
    df_no_punch = df_no_punch[[
        "clip_id", "subject_id", "video_filename", "class", "hand",
        "distance_m", "start_frame", "end_frame", "frame_count"
    ]]
    
    return df_no_punch

# Generate and combine
df_no_punch = generate_no_punch_annotations(df, PROCESSED_ROOT)
df = pd.concat([df, df_no_punch], ignore_index=True)
df = df.sort_values(["subject_id", "video_filename", "start_frame"]).reset_index(drop=True)

# Re-save with combined annotations
df.to_csv(ANNOTATIONS_CSV, index=False)
print(f"\nSaved {len(df)} annotations to {ANNOTATIONS_CSV}\n")

print("=== Updated class distribution ===")
print(df["class"].value_counts())

Generated 866 candidate no_punch clips
Subsampled to 250 clips (seed=42)

Saved 1798 annotations to ..\..\data\metadata\no_hardware\annotations.csv

=== Updated class distribution ===
class
hook        392
uppercut    388
cross       384
jab         384
no_punch    250
Name: count, dtype: int64


## Section B — Clip Extraction

Use the annotations CSV to slice individual punch clips from the full-video pose files.

**Reads:** `data/processed/no_hardware/{subject}/{video_stem}_pose_norm.npy`  
**Writes:** `data/clips/no_hardware/{class}/{clip_id}.npy`

In [19]:
def extract_clips(df: pd.DataFrame, processed_root: Path, clips_dir: Path):
    """Slice individual clips from full-video pose files and save them."""
    success = 0
    failures = []
    
    for cls in df["class"].unique():
        (clips_dir / cls).mkdir(parents=True, exist_ok=True)
    
    # Iterate using iterrows() instead of itertuples() to avoid keyword collision
    for _, row in df.iterrows():
        clean_filename = strip_labelstudio_prefix(row["video_filename"])
        video_stem = Path(clean_filename).stem
        pose_path = processed_root / row["subject_id"] / f"{video_stem}_pose_norm.npy"
        
        if not pose_path.exists():
            failures.append((row["clip_id"], f"Pose file not found: {pose_path}"))
            continue
        
        try:
            pose = np.load(pose_path)
            if row["end_frame"] >= pose.shape[0]:
                failures.append((row["clip_id"],
                    f"end_frame {row['end_frame']} exceeds video length {pose.shape[0]}"))
                continue
            
            clip = pose[row["start_frame"] : row["end_frame"] + 1]
            output_path = clips_dir / row["class"] / f"{row['clip_id']}.npy"
            np.save(output_path, clip)
            success += 1
        except Exception as e:
            failures.append((row["clip_id"], str(e)))
    
    return success, failures

success, failures = extract_clips(df, PROCESSED_ROOT, CLIPS_DIR)
print(f"Successfully extracted: {success} clips")
print(f"Failures: {len(failures)}")
for clip_id, reason in failures[:10]:
    print(f"  {clip_id}: {reason}")

Successfully extracted: 1798 clips
Failures: 0


In [20]:
# Cross-check clip files on disk against CSV
print("Verifying clip files on disk vs CSV:\n")
for cls in sorted(df["class"].unique()):
    cls_dir = CLIPS_DIR / cls
    on_disk = len(list(cls_dir.glob("*.npy")))
    in_csv = (df["class"] == cls).sum()
    status = "OK" if on_disk == in_csv else "MISMATCH"
    print(f"  [{status}] {cls}: {on_disk} files, {in_csv} in CSV")

# Spot check one clip
sample_row = df.iloc[0]
sample_path = CLIPS_DIR / sample_row["class"] / f"{sample_row['clip_id']}.npy"
sample = np.load(sample_path)
print(f"\nSample clip: {sample_row['clip_id']}")
print(f"  Path:  {sample_path}")
print(f"  Shape: {sample.shape}  (frames, joints, xyz)")

Verifying clip files on disk vs CSV:

  [OK] cross: 384 files, 384 in CSV
  [OK] hook: 392 files, 392 in CSV
  [OK] jab: 384 files, 384 in CSV
  [MISMATCH] no_punch: 456 files, 250 in CSV
  [OK] uppercut: 388 files, 388 in CSV

Sample clip: s01_no_punch_1m_left_c001
  Path:  ..\..\data\clips\no_hardware\no_punch\s01_no_punch_1m_left_c001.npy
  Shape: (25, 9, 3)  (frames, joints, xyz)
